# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq9PWtz27aW3/UruOx0SrYyI9lJmmqrzia1k3rWSbyJ23v3uh4OJUISY77Kh2wlk/1b+31/2Z4HQIIv2fHNbqa1+AAOgIPzxgFomubrpEiymbGCHyu3jYNfjMgLQ884Ojas38p4V3rx0fHBofHIOIy29L4Qt0WZeXj5IhSxLzJnNPo9TwxfGJlYllme5LORYXxvXCR+khsh/B8lvsDfv0rhe7Gx9LK1h+9EbEBbr85/Nzy82IqPhpWKbBWExuvnfz99/dZ4BJAM4/nZxVt7bHh5YMSJsfF2hsihnIcQikwYZezREIwdgsmDdRnAC+FQN86goVRAr4xcGCl0EVuPZdvvnr82rKW33Ajsf+qtgxigwuVZEJe3thGKHYwxoV7gCLxsuQm23PXUy7wQxjUzJNAMOiLHahRe5kNXxLqE6jn35CQ0isxbeB/gtYjSBKobWw9BPfLF9lG+iQzLD/Jlgo+gZ7YcAKOIep8UYi2MZRJjU0sv9gGP/6omwrB+hXI/UFUokwFmRExdV12lmjqKjLWIAZHLIInr2ajxlom8DAuaK2h+XeKgqCBinNHGXTVgNDAx+QZxF6rZqNH2ocwLkSOmUlEEGRaJgjySBUVuBHFeeDH8JxJnZJrmKEAEFdhhoKkM+uqsSiA8KAoTtFyp9+tldRUmC3WNPQmD6jZI1NWHPInVdZKrq0yoq3xTFkFY3e2qInm5AOwvRV49KTaZ8PwgXlcPgqiC8zFIgYrFSN3HZZTusOdxOlplSWScn54Z8t1p5K3FmH/epvmICzhGjYFVsB6Nfjt5d2LModdO6hUbxw+y2IuEpe69RY6/lutiw65r2yOYSvfs7a//DrWqzjpnyfLaskd/nJ787T28sExoLS7MsWEuvOU1/oZiRfdZsN4Upj0ajb4x3gMhZwFQ2wy65SHR5MY6AD6CWTUSY+WFOQ4uMVJgcSCJoowWwA8iBLrJtoGfZCManwN87Z6+fv7qxD0//fvJGXbh6WN3Mpng/yN8+/70GMd5OHn8DJr2xcrIvZVwqVkriNb2jMgKaOQPL4QuAdPHSQZiK/joEV1yB3HG8K1VeJEXJ2NkTPi7wqJFYjtIYwgnWEF1IJGcKXBJbcjZcOivbBD/ZV4AfADtluIky5LMMoGnpUzA0QMytB5sqX+AQax6MzY2MCwA7uTBR6HavjF+No4ODeQeutrX1lkFWaCMikAeAmuCcAO5Cq1aURAHUYLg0ltbNktNfA/AfzH6J+ABDa4zEDyibiDybi0cng2NqBmswcKI5biXSbqzbP2FU2yAUGIvCC1L1RxXMGw1DWfP3/z6j7fvqwaxJspZnrkYiPjdqxdIsvDz3LT72463IissWYRKZAIESoyvJaEtQ+HFZeomZZGWRW5dC5HOn02AdoLYXWVCuOvFfDqpKfAFCFmPJJwmKQFNxjYQH1BoBqStKtRJXik8g+R8RYRFtqs77efQ5xxYX/iWjwRr+DhIlG8O/qlY/kMSxBaLB+ft7xfu8ek7wMH3pm0jklSpIAdRYfl2jXgcCTZBsg4ESX7tljlyVxOW7VBBUP3ff380qWrfbEC+YCeBCAwLNI/l5zjziCykY6r0s44ybUJofNC2D51LUmtiN97IHmURKHVh+WMjWANrC1cgOebzi6wUzQraMMVtkMOU+aD/TAfEr9lqFf9B2UxEyVboxRqlHo4acbsUaWGc0A+o07r11AOlMRotQ/g1TmIwMQS/RJJzXWDawnWtXISrMWr41fxNEoM+CJP1PM2CuNAGgoWcFDqI5RDZslu+KMSy0HiLCgIEKAp/m48BBQt8gc003+QbLxXDb9xou+dlsbfiwOvUgwEOv3GvgxjJRbJJ9fJDskAm+WR6ywLNMXNmAJeaInaXSejh3edmBRcMP+DJG6j0EnRV3dY3oLbRFgO2FX6AHGrhBZgnyTYIbbKcQPqBbgd1tgRzuJAc3IQvgKWxP6ReoP1nDvZH2spwf/SEHizYUIMHTydOp48AxEVa1rR8P4vLLjtozWgE3JAietecMvW9QlhYHqjC860kBb5ttmprEmKYmCuCbrTBpiAg4HOb1Dx/18E5vZKocNMkCaHEcuVckI1yDvcnt+BIgGNigWZxb5LsWgD3H4IaY8q4NOUz86ruMqAj8q5h6rK8gzCSDW5yLSVIzXwizsGkJERoXAZS+X2Atssy8YN1gqYv0FmJFoa0oDWfZudV/swYXkZpiLQAbypwXpEFCxgO2EelQN+BaCpGJwBsb3SKQHuAFFgGHhnGucgB5ZVuUCKTZ8v1gyWIi/qNk4sCxuKB9rFMyYmgCpGh7L2lkCfvKFjzICpXc6gYciMU2MeNQ1UVX0J1opGhcoBTauHL+GuwVUXzAPNLOG0IHpE/9m+wRSCBNLkD28Vh0CkBCs531gLe6vxigoLPqWRLs142i139M4w1YvF48E/8Y86o2A3mGoWPeyPQs8ilygOXMMnnMHwBxndcPNLCDibNcedxtIWpaLDrGSjjogo3VE46uvTk+RPH+eKDl4VV2AG81gaPkde1Kdegn9crbyncTblQPlgOrrZLswz8l9VVwBJCPXR5NSyD0X7L0H5rwrBshwbeNVJgyjN65wZk9w0Uq2CLLZVykJFQcAyUrfv7A3Q4LzJr5ZCnSLKfQK24uS09z68GtQH6saJH04ClYZlvSF6mJfgJoRAk0Kr4ym8vZzClLRtOWuGTUc0WQCuZb1HHms3EgOxJ48lNAPqStBkWR691AU0sytVKgO20nk9s7O2qixM2Y1Eh9ONrAU2xArOePjZ+/hkcUru3oHQgF8NolyOMewvEOB9oRS/sGgNoEmE4wcE/mllHo+3n6OlTGqm4bfajSAoP9WteRpa4BRWZWhkY1jTDdtdetFbmOXOMjGnJMB0wy8z4xMCk1TtzpqvPxqsX+NrsDm1lftJGYBwYxcyZrD7nmhaReCGomlJm9ZQGqdAkxBj9+bEBFRaJRhTMtLsjH5xUtjXRW5VcW4mM4+DiZZjcvPYKEAzx+hxAh4FmcGJbgKO7yjvYGmqsAlzlGLyzFh3LXq6SECTwnPpb5sLF8AVIsbzrwgDp0ICaU/aN8RJchc0fx4ByX6ANsgqWFCDl2Ox/TSe35GRmXhrI2OSg+FHDc0TsgW5wVwh660dWtHS9cJ3MzWjZ8oEku1/sUo4DdAH2tjLYUpdt7iFQuoLFqNCCthO4ZmkSB9BQj0i5J/wvhi0JFkdZ0yvZ00yoN+AegF1FsyzvyITSH9CE6w/+QsOy2JHX19RqvyID6samY1xgbcN6gjY4PP+h6reNIRowWTHKRcFY6Bq4hBppUVQWrCcj90IK21bBaeNXQBGGmDmwYxRetAigAIXrlXULwCtQMsbNYTfozYS7o7wl2YtfKfzlN5SsCpUmoKJraYd3DoYfwXbJnWXpeyCoiqgMHSD45MYtVkeHwJ+ItD2V4vhLiy/AttiAY3fdLt0QK+wvy56/AAjrLAHEvaNIQqYzsz6f/RZao8S8Mrug324Su+u0BJsNJg8VSk0td+sFaFtz7Xtbbrj+nVE0YdVNU5Cn4eQPAydOIpoFqjL01SNJtUx+JLU823GcltzplSqtAAM7QZqG6DcbB1Xx0D9zI6v7B35QHGwPDyYHhAAQAN2w0wPkC6PgLsHVQyINumzTWHN2HjQ3PCvgujK39sxLIzR0zwnoQ2jXx1PDibY0kmGyo/jRA0YHgCXtbb0soLhsXnj5A4hPRrDuNXzwT76cAnuJMNp+XSqsEbKPElvTYjVIDqTTwMTY3Vl74Jw1Z2sfbd49bf+3k9Y3ZV06/ydmqzFPY3StyhyVbJ6AAp7S8unA7HFotZoSvu2dj4a2K8Rtnwl9jtU7hjP1uKRYrHWp4YIa06XoVUutCMAPuM02WDD99cyrjhedoz9O63r5Hryt+oiKem9Yn76j1r/DzlSXBJE79B1bNN997qG0vXbv4ItOoLsfofs9iw7R9YvchuvR76vebdprVGTy1EkMzfZ2idYOvyoGegdpj/aDV+sF0N9+px7c5uuv5Y7cR5nvX+jQTbeKP9H26/JovUqQC2AiCnyCaxl8ZCcZjUh4qAfEOw1cmslqhd6KeTW7o6PKhyND312mpSurtpw55Jo77B6ZLnT6vIkvw1qCBMtwIWsHfgo6GHXmiG02xBlipM9S7gbaUloNbsj/cVtd6Q/s3vBbqk8DSdDqiW6ODrjATpFYJqDMtPvXQhoGHb258bKoTK3eaMxrucwQgirkUMydYRVV240EOt+Voyqb6S52/B576MB5PiblxAHY6ODVcUbTjJY0gpDd0GuRxTDlOFO//n783NgBAQTrejQUfA3XSRYUEYdfl+XxmzecuQUw0gx6nansIA+TnzyM22LqV5gs7+ctymBNwzFoGCU66VBhqfSUY8WuIHmZQe56Ww+GBxRptYiBnf07Xa9uwLcvvec4826abB8BPM5uiMWNTEsYgy89PRwb8MeG68nYkP/ZdmcFvQLr4B8riFpFHBGGQZoLy5r++HRsPAYwR0dwcTiZ2BT/C+cWXI+N6Y/45wn8OXzypN0QUC0wZ7wOEc5PUH36EwKaIsQfa0BP4SHmSUyf9oJBl5Ox/orSzTBe6YttsBRzE+fBtMHfj8HABOEGor8VaKWgJ1cP4pXIUC+QZLKgfX7ulUWy9HJcikFwY8MvQMXN+SUKrmL61O4PjFmU4TIPojEmabl1C3kh0nwOs5EsMR3BzQTYXCXqh/n08NmArYgglmDPXOdzJbH41rway1w76NR8DVApx8SlfuLyUCTyTdu4rQk138XLTQai86Ow7NGA4YMhHDCVQWkiK99HVHxRZN9QwCNMHUyMJAqKwFfmZyVmUDe5RSITtqq0higlySv8+aQpfVBDJJhOKlONKI+UJyHwqizU4+BiDMJ9GZbQJC3CanYBJUryv/eCJZcubjCvFFdZ2Xh2jJO4CHCVzIjEhwTXZtYhZbFhsDdpCqAVTFrhFUUmB6Iv0t3PmIYKOK6WQQ3D6TemB10jqGFYBeFKZWUN+erQZG1naS1ZgGuyqQ7ku0d1mYPt1JkeHAerVZmLLIervADWFr7WAM8iyl7AB9/YTo5Lm5Z9OTuaTK5aiVeqN7Iw5tyAxCjD0Fgk/g78OkxBzYQPBBKC2WEsqrCUKWkFbBELL+xOEJaytvQYrEtrrPlaLRokSTE3HwGlFGhFXuDL96/MFu3xU8P64/n7C+ORQffG81N7JmOlQCSUrrUKQA2jfoInOTxkz+tuZdVDQPpK8LC9J0fZjGG2VQ0uFcIwVRpcvuO17B6LbKeSweIcs+AmjB/7PgUbC+RYCwaRL2EURW7adpf25TQ4qaS6vLpSM6RQJbHfywgEapEFYLNowVe4f/f6xaumeminy8g+1r6EWnDu5RXsg5vJsKiE33FGGvBvMNEQSh1MnceAAbb3WIuJrRdaQ62Q7dcY8d3tVGRLzXSmS6m7pqIbkCiK2MmUNKwIU/+RwCtKr6jftO+kxftqDjXyjt/T7VdvqKMW88NLx5xdVAkDaRQ2pAHl9JJGZ8ECyhgUCab7NrJFmYNFxMnb+zlbPmTN3Vw5IPHsynxxVRKmOfUymcjcXNFVLTpvvEj4F7w3INu9hEdWXq5Wwe3cdNJ4jYaNCEUh5hxgoqXtFrtTgrG3FdZq42CCuN1K3aL0MvluWLKAyU9ZhlqX5QL7Yu2S5zaPU8fLMm9nXU4x90b9uQKjDNnJjUUxb7GYloIaxODJtRqt0zNp6X/0QPOxR2fsMSFbXai0FnZa2oaAjJbx1mcpYrOYCS2p7BCM4L0BRUWAbo5m1ZyY11IPSXGo61+MJxyf+tEBTwBmF5PM8ktN3aqVdCZG54J/rQTKOGAWg41pwXRRE0foWuDc5agag3WZYEAR5zG5nF7ZenZcb6QB+Acuda5Z3RlgbLtSvS5TMwDp+MGK8FsEFInIKK1JZA4OTN4pznoNj97Rk9FAOEPmdemDwYlWefjDNbihqoI06flpK0SpVZbdm2s9s2Sal+5GtMF9UfxZH8ucpqTbG5y99jgfmsyzxEVh3tLBOypQY8gEvLYLoIIPrVBD136RkjGnTTc9b5oiF/9t0W/mCs42yDArE5tKsp015HPyejKVcXGBeOlhZr2t8lgoC2MoJMB8NxnyuHDnU0mBk0/byAHL2W9kxzzCp62kmQknzfzPf/clzXDiDPpxn9YSgiSS77aZF7nrxXdXWN38olTdr5HJp3b96Ml8qa5cWXK7izXnuDTkA3oDewRD0zjA8q3NOOBj2B0tV8VPKPsLNwa1vY/mtqFRT923aY4bB1ZgLXgxmB55q+jGA2oI040nd5HQppP5XG40odBgpQax4iV4ZGPj6MqJwH6zjZ9BC0z0gVVI4qpJUbfQN756zd5q72HRTe/9u1x48jOxBIYC/gVaTBYfBG2WRP8rQ57m7YSl52ee9K4jcD9FLbIRAb0jrUrsQOfd5lwsTuKPIkssD1TXdNKI5WKm223ems1bMB1up2NjB7+7KcC4zRmDY7oCpQpXu+rZTj7rwRghIUtSy0KYO4ILbueU4MKvbhP0bMgi9saHUpfzrqbvwa6ZPtMCrH/1x+7QthyM2+V/gXIA24Ap2rJy48C4ASEEggGrwt2G77quLlQFrYEK2JpODh+DmQV/O5ujSBETwg8fs9jK/9KM4jQMCjffCFEoNY7X4N5luGLEuwDHuPVvjF74WG7+a3jLx8EW3GDa4bZJPpDLwCu0vBs2FhhGydFkpLRBzEJaJbHfCalQy3sFgraPIFsvvCYrDHFuaysRtdHZ3CdHjRKFkaTJEHoHBtYQnzWYAPtVcQGhvSq3pB0NnuPFOz2WmYt1jjGsHJCIPfCygnKH5XLxWFMzuJYRjA3K7BVgbKLliXuQQrsTANhWooQh96xotFrk3LOgtZAjUcXg9oPqbOjQR+h4KeDSBwLH5qAdu8H/fZD7qqGcwPFqtf9GSCXvDizVBloRjzkvElMgBB8hoUE5YCw0lWGG/gaMPHEmj6/a8giLk7CekagEYjE2JcjLfIYJbbjDFcxksNHzTi4y1iYGcoi/LHPcDomo3pFXEkAP/gamQGyPScTAA5RJ1VNOww4oDduLQW/Fdt1ZdNgIVKJiblQ6wdLtPtS1krJo7svBOggKhA6LXRsBfAR41MCYetz2isriEt9eVTYwKn/kAE3YShlOT1FYkTfRSYWQDA4Qv4pZQrGL2mNBfzVQoTb0FMa4OdXFdP78vlnD1rGQvp54CfVkghwOBG/8cgk39zTUX6LDJTIJokYC9gyZqPHasi183inV251O4UHbeoflwJptvIyQkKpXDrop70XR0qgRMKXvu+RJNstS2w7KSPDBc8a0w0jugMB3Qbx2penjl0CkaIG7qrq1pwrOpgvWhAsOR4DJDa6AOdtTZRmCJnA3CTArKnC6QAfpcDKZ7GnHTTJc02BCcZfJhjzPsO1ToO8RYfIoHUjAmLF7/ZZWVEibzraPHhEexS0MEV6RpRMRLrUHMpI0b/HSF67eVHM4EGmTPeuJs/Wmt+wZqhouYBisyKGZZsiMDZ5qZxXcupyDk7foe7BCGKrp7pTvxc/h/nwFinjxeRi8vsXO7WFDiVUyhZQlKgGNA4xf6vezPgrQBIlk41pKga6f1xKrLTUJ0a2NElLY4Sa6vApx8pJlO9Qpt0/gUQjg8DZdNI4Go2sjA6fs3Txs8aKXXuTo9XV7GaKlvl/KgyiuhuK0HYDfqHB6ldGegDFoMCvTFD4SGFvzZrSDEwPuHqYzVEtqo4GNOjwlXRPni1P4VP9WGHa4M6KtrP0aQainrC+YWbsdBCcAX0wgHQ++LxgTlWGBy4yfrmdgNKJVcS0NVmrJCQoRAceziapRSm2JqBxTNqMIHjLPtLajgQI5tXBPCg88uKo20si8wv0JQFe28WfbilZ9kXmJrS7QjjmNbnWQexpvtFxctTNYYnFbWGmdt8RttfOPavrAvUSELk5quTPRSRNZOSUv99Ur9lVbUc2WRkECQj0IvjhF2vEedNQzm8179cT4eQ4+EePkycOyuZJiI7JWh5s4bXW+O5be/C4Jt5Xj1XiKqOrP76JiPTle1fanejlwKJlDRGmx462gVjuKw2wFnZSzTfhrCsj/n3WRfyK1Rkr6RnbNnkUTSUGDNnU3/wZr8VN7vDflZvSgxRe7udoz3LHeFJ6edRlWKDyjX8XtkXH3Rs6NOshAOT5S4KNp+dAVm15NGKwqmMa/zNsWAi45KAOhxxTtrilV/dMZgfzcJtVfYao4P7q+Yn3DR/8MHFtFioc0EW6fv7b3bNH+xviD9xvUeUe5Ya0yOiDtBzxrzguBbR3HNjiPEQxa3GlthInh+WAlira3XPnJHkUSCYsVJ+Qq7oDXpPGYy/EWiOfB9r06mQ/Dt40NFGjvUz+BvMe8c4AH12Pq39n5Rg9rb543qHwd4k7kplak7GuxaxowSVrorjwKGnnaG8r9qSZQKxqhs86aeEM1qlNHN6q2NxN7o05UuXaAXEhe2sOFtqATFrsCHaDL2cz46UcNg5vG2Sx+GaW5hUMc0wlQLgxfbhruaUhif+NsxK0fgF9cYAPTw6vaDMzKWKKPA7yUYqUOGAKGkZfEM/KaoqB83Y2ounvDxe1VoHHXcpCNsPSWN+TsyGsgYtUPzP+aHh49bkJRQnpOJ35Au3NzU+J6ZODjERwbSvWZT51nmOIEoknMJ1CwOflQNJmbv3pgw3iZOeYHqGBU07zCrroESC5wObBnTFKUyf28mQgDscUWn+mmNDnw6ww9+NCLFr5noJCeGRIVsYtRxsALe5BOPZOOPxUFfl4rvPkgekPVLfbg5mozjdk+NUqdddOfWU/J6b2ipZHzf17JhlHbHqw0OaZZn1+adN04oIemXJaUN1yUb/Sy7FxzUb7mknStF+QFbiyGV1yotQ+imlxZsL7n4tW9XklSgBqXvJMj47tGNyat7HA9Z5Jm3gIqPATt9B9lUFBaJ62MmHo3QRrxGVKk8GZapFXybsVctjqtUS9EfKiX6dotrCP1SsTBTcDMynoheqKX+ty7mtMvLnXPkK1zfSVKrsbUosVWjmOv5a6L7H7DpYFOtBRAzo/aECqjYsD46RyE+JKSQNW5ppgzR/pTmz6U2XiuXbCUbDBveNpz5XCzrKO/LOzg/3aua7/FWQm+2ltH6Qf/V1KPfyrBxz/jAfygDKS/uvyrrmohKH/7wdSycZEkoVXd2uNKNPILvrFrGckmt7yzB6GDCFWQF4k9VgKRnvG1XQtCeipvbCUROdhVwb9WIgwtC92m0KnkuqJWPruG/MPGeYr1SyCySxMegTwYss3eqcMhMUhFldDyuu5NZmwAruVI+2hKu7XCs/cYK5JMeZGtSDqZ3/7nt9G3vvvtb9++/va9a+ImzOv+g9TIsGudntY6rJEO9Wy0jgfV4vFfeVH6QeJsiig0bdrwoFVGNYbdlhoNRasllSOvTvmzVvgDj6ZqD/ScT6xyCf3joVFgkcFh0HpYX+Rq9oU24ZbTLpspvNTyyiQ73P0Uf6YsTlvTDyeKHIV2pOA3xjsMQ5OoCY3pZGJ8y6d1oMeBOxnwdGlMyUJTwsuFOjwbj+PNgiTTTr1A2ji5lOemoQ9lnVxW56ZdSWcOWVNmOiExnFTHiFGNxzWuXHUMAK6jwquJ89MED7auG3iEbWoVWEGrilWNO3vBcBpHQhXB8toCB2aB8nKJy0455QYlaWuuTNN8vvXijySxF7hcb+QlzE1EDp2PSS8kxOnYcEZftvYaWQqEvOnAlq/6+Ciyn6ADmDWW4wJalyxuZawMBv5kbFitHTlTHClljdKIpj1uRGVBeDh7C6jl4YrxLY2/J4Mfu3PjBQW0+MRuYjAFT0l0UbiK25ILYDQOjT7ZAtJaGKBE4roIH0llFZg3VMxxrsYGXOfzgSkDDvfAoIh7TnUF7qEsAOseq1BSaq7iVuHeBOMKP3lnsVO2S1yrYa06bfSyOs7wCsPBU11bqHO2QdgvrzG1sJWDp5mARyAYT2LMEPFm6ux74Fl0RHD1IpFH65itEJ1qoUfBtDt3oHeuVUqdz9gawiBy4608Oawh+khkavJyi4ZS3ucBk0k0xEEqUkiBMV9GtolAAU/AKlJmjDXpct91d5PjoWhmV6ceHR2byD9m+ygQk/ygbR2GMQd2mXf/Ebj/+W8+UcNsrQMAmPt2l/XeTD+nYt+qz0OOFmnZkK2VwJ6ZqyR9S2Dlw/KpmjDzLIjSgNCPOYYZz4S3Lj+ILMnNvpU8kIN8uFl1gipYSmR0c9iXFmQtjAxhArI08eSO8dPn9t7FRpUVwsTWNCrqxKZV4lJ78/Z67v1PgdibVqKt+vZW3uBiGnVbRgf1JeWf58bTCf5j2rprBZnL2gPtOOIWe9S0VMjSM2kOCA/OOlyY9lc7uo0nlxCsL6v3kcKhjQulGQ0Pd9ulJa6zw0TnuLPTw495+FA/Nyw6bUxFPfGbIGAaAe0k9p00SobIhKSMSYjE/aBkE3yqKGE2/gzNfuJ1fLimxs3e8xW6GQA9q/9fhUrumUIwQLnUx77ZZzvV9LJlAPM0NPeDFn9f9ZbB3zD8LSY0k8vokw+MnXrGdFafdrjDA5HBuv1y5VJblQOHtvBLH3pxLe1U1j8NIpmO2YolPVSZrF8ohM3zIC6UPtLj9KfPv/TUoYayUOs+7RWfPsbtPapjAA9DGRCsGrjMnYpBwd1HcWzuyBWh4CO4yA8lviFI++lQTWiTEhUdHs5okmi/2/3VwODelXsvBd9vb0djj8f9bN8+U7BhMGqeHDp5T5rO3Q/ykbQORl1O40gJkVGz+Ya/p4GuH9bA4dlI84XV141WpaBTnEP14SOMGJAFDTKCF2yb3zaio2QcDdIxHWyLuXRJXn1yagZ+IFgVRKNedXb0GO0Mssh5dxZ+XIbjTTXAYjF0HGOctiMVd1Cp1snz+vBOXqvj8zTxwM0A/XueuRmlKXOPjDiQvaSTWXZLPVWZCJ3WU00eI66RUj12vCUIzl1RRa4Gv56gB95ahZRVJgupOFwd4c4xHMkhbjwUnk5sB4cQrk3slA+WGV5z56pN7dT/OuIsOlA+N51cgcreku5xkwi5LtAz0SC63dqTsXFrjwaMBpR4YO2DD/8jyDq6Rx+8UZu88UZPtsM92XZ6st3TE/UOkUXl87XOIGccFAIuoBgXHXQBcjLwKCCUz5CCZDSOv89keIssyGrC0mCJWH3lQrmloaeYA41w5+vE/6rjxq2G1bCHQwD/RQmuQXcTfDu8xiG9auG3HdvIlv28ubrr0zJQsye+o6MBSrTHWW0b1Q/md/JyAVaw1fXTZSncHpCOeV1dC/CryL4Mg2uB+ypc31qeVPH4Ku6OE6PEAHFKs4KyLiypHyubWmpIfbmSZX11iy6Evl1j1flYDJ4pJ5m7Yux2JgZ+dU6KpDdvaZnFo9P66UN/a7R7QLyLWDu/GM8OidHTbNkq34DHiVXQjzT+cXpOdO1XdK1RctXb+03U/9VkDU4Y42zM8qTtwS74Kz8r4fAnqPSjqdxF2DXUFrqyr+O8mk7Wg79KJy/C4fyZKnXBOhkbPV+WgSHcmPaXb1lVjnlzA0vQ0a3suwLqMb0saX8Vp8vIQTtgLBtpfR4HitmNfqhUDfIEyUec427iMLds/kZH5TuivKNMPpJMcxngUmtinht4czaSxtUp3C4ZWXM6/cYqVChlOhBEqmpJiFU9ggm16hJyNqsSi/A+YHHTsqrSpJ8Jw++uzfVClF9+nHNOC3/cc35+afIVnYqVlviAckBrfNc0hUiVZBWkTErIcRjGx9O5OK3B9fJlEKg8CbnXfo67/LQVWtys1aUbiR+MVCBkhO/cZAHMM7KW3beS/9MzDONGqQfCkd06kC8ayX0kEi2L6iNjLUsJd1tR4x/THvh0koZ5hqe0ND6WUK0v+0E25+pBOv+Yjok454yna7GbXytZw1/MnK+26gHZEXM2J1rGCjgNQb5xWfZaUNO+R07/VgqfsuhKn5pft/sZlttss63GeHew5j5G39oDPtq2D+jWHvjER7xqfKlHOQn8yR8lFoItS4Hm23tCBEFBNS6vbOh9gVshB+MCnksmDvyAGq1a1SAgKuCttKCxrU6RwW5dmpUQYFog3aBaUe8GYuT1CS6IzE6RBlN/DZ7WmOOSpC93Nl51T1WvvhNzDxEA8rvvaJ3KVthIAWH+Gf8Z47LCtiEp7sHsD9pOwvPREzgddPiV40Cu1VX78NGWCSs/u8jWcN+XF3WVutqzGr7a0sZBbNQF9RhiZpLVkC+aLdAJS923Nw38qIQJGGIm8qGjUFuLhliSnv0boilYRqLYJH69v0qfOe0UkwilO4h4FO5wYzaX6uSXf51/BCmd3wQliMDH9ZvTc/f45OXZ84uTY/5kHIr9EGy9cD4lwvvYTaEiZ0l+FBQoGM/wwjQUsrm7845AsZNai+8v3r47Oaapc/ATGNhZyzKdKH2Mbpc8XspUbhgbt/gp0T/7T5LGeEHfiDqlP0pm6fLdCs8PrTHAifR410jjyAQY+EuJSOQhmWP8W1DQGbMhHjeDgQ8VJzr4xfC2dHASqGoHMIrn0GKqhAz9UP3Xp2cngJI3J/gF4k/1KQfSOZoZl3WQmpYjTQl+Jk/FUtFcLtRv/TT9JwePdq3BkGVEUMg9V+WGIEHnb3fVF80I3DMdHDDasgwJHpcYBETRr0wd04CQjprjowJ84PAglMhL+UPp0k1BMHjGbA1GvRhEDQcVfLXm7Xp49LBPmHqCHTrhAjiiV2cvDPX63vBuxEJSs4PH0poIBI8rzhL+Tg28vz8wMFac1eKWgLU69/LF3+8BJw3ER4+w/SOSwTnd8mfTS0D4fSCoif8RJ/41Bqgrcrj3OMIEz9Z0nj0FEH/wNgaRM1pwSe3s7bF9b2BSRsiz38zn8jvYNSuatrZQIt3YHs76A180+ErkS3D8+zvSS8GHGpyKl5iGh6DIiCmRFUa2iBFwHO9UwBd7Qp2mY0CrcvvhrQOgr6OnE5qoDrjq7QAQ7rHC6B8cCFGsKJH5uU7vl4aLOofpoWEILezQY1qpnPanY47ZzHHbnsmRIJXXXqWyUyJ5+1tWywhPrL6UaYkvzk7eHJ/g1zUPSNUcHKy8ZZFkuwPK+ilTekhx6gldnnfXyPoCjipwg6ya7siQOzjgBgKcXUIP3ACGOJrVgQovobIp8Qd3jEKTzoux+IbBMk7lC76x+8BVoRvGvXpC6eZcubpnwCq5nF/Ku17Q1ZzhEKcq/YQfyRQURt9BLaKpt0Mpt1BSxuxriO3dXXUBMsLqdnqAEbGYkmioI/XHXiV4zv38l3mld6uT3KslB9VCvSRBh1dQoVqJk3ciG2q9qhc1arOCzqVtfrgzzfi7EfKsBeecnAWgXByAD9Qy11+enp/Qc7BI9efvL47f/n5x14ouxmzkpo1FuaLzKabaR2ExxWreOgwIO4wWYJo53JtOdqr5nA7QMVaZFwkKsFIdWZcT6tjoM1/iiX09VmNE1jP4pejBWlkDomH96f9gw8whtF7fO6riwVIo8LdnONF3Xon/fr+xrmVhsiTlSjpHYGLTFjQHD4VOramN2ZKP6QD7Hind45PSEZdxy+XhQ4nleUOEnEweqNNdAZACjb+DNGxi6Ogeth1Uqb4v08uGrGY2RL1us/PuWLppQxr41kiHGC5R4tntuev2UB5JZV2L3VglnzaPp4ryoWALOC/GL3OmbGwIYChcDH9WRvIBHZE0vAVXI5x2RizwCuXBamyfrItbzIgGx91k3UHE0AgWp/gpXZSkuN0OZdNkIK2foV0eHE0mk1l3UyS/Ho1G7smbV6dvTtSEjEaouUFeuSJe46nsUHCeZkDoEn3rMFl4oSGrjWS3FJTOjpUa/EkNr4pHyM4oYP8LeBU8jA==", "blender_rig.py": "...", "app.py": "eNrVfdtuI8mS2Lu+Il2NsyxqiiWSkroldrMX6nsb0zNyd885hjlCbbIqSVarWFVTF0pqQYBfvIABGwfwywILG94DGDYWiwOssQ82vG/TfzI/YH+CIyIz68YiJc3MWcM9GKkuWZGZkXHPyJBhGG/DTCQz/pldiCkzXyfc86Mu80TARJrlcDOCmzBLInoWRpmYRtE5i5gbLSMW87kf8p04nwa+y+FZiP9nCU9FyBlncLUSKXzK8pBleQgg4Pp5EOXeLOCJsA3D2PGXcZRkbB5EU339KY1CfR2l+irhoRctizuhr1LhJiIrmqX5NE4iV6TFk2yRCJhYOC8e+EuxU3RMk2Y8haudnVkSLZnN1DuYzcyfq4cihNkK/WouMkc+2dk5+ebtuw9szCbGc74EjCSGxYznUZIIuvrAg9zj+jKTV8+4H8ir9yKO0giv3vEwSvHiDZ/iy7Od0/cvP7z8iLCvdxj8ewBrsJwmYsTMmKdRarHIzRIhLOZyQLvFMnGZ5Qnv2mzQZ9QEFmspAC7AWEawihH7mCfTiMAZdGkAtEOL7R8dWOxhH/5ZbNg/OOpass0LkfEg4B61O7LY4WBoscHhesP3PPZlqyGAGx4+tNjBeqsTwAC2OeirLo/W27zjl/6SWh32Gx0e9I8fQrObnZf/8uTd6dcvPzgv3r6HuRl7SHpAq3tv8vAq5+H+i95wj6cpkMaeuOTLOBAOAJ2L1NjZ2fHEjDkzPxCp6XVH1GuUeCKhVZR4cngI7b3IngdTXBT1FDil8STx5/Zselk+saeBCL1KC7kq/ucCGnVY/NPNgOd45ruRHU0/Vb4uHi+zoPIYCDERqR+F+56d4psG0NgXn3mqxxoLoAV9wxPXhyXVtzABxw9nkY2MB1SHn8+QYCbwScyzhf0p8kPTs9isy2ZRwmbMDxW+/BnTjcSln2apuf5NtwD5FcCclSBS4CThmcj7Nv5Y+9bYNbrdbnVi0N8MBRF+DgBBJtDdDNjTSy/8bGGahh1Ec8PS3YJwyBNsrJc9zZdLnlyZOGW19hwmSxgAtjYN/ilPM4GceH0jOw+AzwkhxVDMGVAp4BAlGvx8ezJiu7vXCGPSIV50kNU7ZyPrRvLm7i776S//A4zGy13fQ/Fo4GTKXsuvjC4TQSqYARdfFT3ODNUBNZeddCzW+fNOt9rJj/9DCwF2zWVTuO90b1h8iS+NCkCR/pCLQGQRqwAGasDWZuXRIkfqgc768ELeWIwoioEwrIGsfBWrjzo9gNfFvisvUzHPQZ6nThaBeFHzuGFphYxnxqsoWfIRu+4s8yDzV0BevFNDWvWFQlpnAPSVRZ2b2rDMtq5nCJ56pdF9VGjDtaxNqZP6jW4Vhh2/7DbELlu7UY3LjqrA3wPdimRUWwB6pBFXQ8hPv//v7KMvQAWlI0bDZ5snVuK0NhlmktTvNqeED4vZwFyqhLRlVkUvzZk9C2gerR9Pg2KO7R9fm52V7wkgsA77iqVZYrZAkS06XeASaJYyU4BIkS9ZHIAyJWzDnGoTXfu8mLJR4zUizLuQ7O6uWiMpb2pMTTrEBSEN4kTJmkKe2DyOAQ2mcVI2AsqDyaBQljKQBEoNypmSh7VugshzSABs7uTrb1/Ak7lIorVOaooDcH+OhLwqhErXIIl9brEVSl05pLLHM9vPxDI1u20DW3IwQpxEBL5Yic2Dey8bjBi2R5EaIgWDdmYu2m5ofqZS0CqyM1r6WokE1KRA0gz4xr5Aqnz5Y4r2KgDElqlivkmnBkCK7tWXv5XPqPWX/xXCh3suDHJvycM5GI1GXesW2jfwpwIsyyiVCgIEpS+SEH5fsWnCP6PkBOwxkL8wV6BaMG/TeS7CtolJiXvbtN5QKz2rYlKVjztnsLCIw8o8sOspB2ZxhbTgAShYB7zbNhCS6GAqoJ2FFuugy56ywcYRnRYawuUB/AS1LFECfAq+gIKjB1oDTkMFO/7yii2//AHIQCT4aWVU5XKnfujgx0BmiFPowNB0un1oIzCS++w3SoUvSZ2nPqwC4+gAad24YYm1UqnQzfpIcB76NSue6mkgW13osSqBAcqMnIDJ2eal/umv/+Z//8/fs+uLGwVI2TjG9+H3oWJq+qQLNs/7t6/JgTBO8gytc4PntHhAMEDD9GRBVyAOyXFJfE4sBi/cnHtJHguPPvgAIy3Qgq9DcKpyAHGjTKuVLy5E4iyyZWB6429AXllsIfz5Ihs/ROMd36vHF34ixq84SF6LgYMRc/V8mc7pSk0ePMSX0hNl+y+Yia6cADO1iyQETqU/S/hSAA0nIMwrUgIkUwxQyWsFIxmMBDS90d1U1IM2owcmLF3cYsAayhe2cV5gj5bLotAOK/LE81egpq4CMe7IGY+u5e+b+PKx56egj65Gs0BcPuYghcMeycyRK9ABf7xOYQaaoP7sqqe8Gt0SIfQ8wJ2bgToYuVGQL8PHcx6PBn3oaMrd83kSgZoaPRgMB/uDozbYUzLde+j45ulocAAfAqAoGT045sce549n0GlvBr5scDVKr8AWXvZyv/O0BVR13vRV6n8Wo4OH8WXn6f/5z//t3z/ZgwZPsdVT1DkmLC9i3fiQTzEqwMlWA5kY50HKWVsHUw1eDVEciWMhOk9fCxBUPEGyQLH19gTAv9TE+WRv+hSNZ/heDoB+aq5Ddz5PAhDRsNBJWvj0P+RRJqjJD8AwMyMbA4NnJoYLbPxhdtkuigywggtCQpIuCeIH9HBmxp+txtf4omyG5N5oZvwZ8cCgaENssNZIMgcij4Zn0r2cSJq4NM49GcNwwP/eQ592fN0k4U6VhMFo+fPrH9TYKiSsmSlxx51r+HnT0ai/8L1sARTW/83jdeKW1DTqNyitjcrWybMDojaILoB48iBIQdmL8DG7THopDN/nQS9L4BOQMp0GcPlZ+RGssBx+g0jrnInGQ4/4b5Tg8B+DHzj3w14WxaMDJNknnIGEmRXzz+C9yMYdMFh5eN5pEKhRp8yj6dF0NntcssFgH2GeTBM/QXkV85AiKRg8iwMUTz/95V892eMlde7sPGC9X/CPzRVToM4lQkkB4js/XUbSWwBeQ4sDxgC6hEUzHzQ9BeWKoAm4FXkoTU3QhThOVNcu90TIPQJnSkXZe1q4B3BZqIWuzZ5DQ+JtT6zg4QpIKi+lMDA7fK1Chw/AJmBLsFBgnVHpgoijABrA43ICNkOtA6/DdOljtAMxufJXEQt9aI6MS3EVgG/v/PNvn5Gq0/rIueB+ZqaZxUIgDXBZE7BARJpWlUsssGtCCygL6GjKP0UsCATZAThWGgf7CwTxF2wZ4VjRw4DWfIV2U6FXLhbAfNSVjlCk2cTAe7CSi1AFPvMAv8ZZyet6YCbM3uzbx8cWNZuBtW+hVnPHYJbDk84SDQr0pKtCqYeNO1kfTFa7P0OHrwyZULs0ECIGuIeF9YTQwaSqjiHhPojCeWK/TJIoQTsZeQzsSAQObaFnIlEZQ4kjE8m4xGS6iC4cPQ8nwuAOrG3qYySZow1EiwekJqPMuPDcW4IaLNWylMJ+CGviZjImd46hvSZoZd5RMxsMthDtFWHC0J/lWRaFthv47nkXhTvgHmgqrYq663V4IzbByQBqMZx1Lh3CkopwtaRRo6b7gH2n5ofWGukgIkkil5X4PELicEWQBzxR5j3Ql4fWZciWYgk0y6tDqtpNaAAZL4nkFDd4wAQfI6S4NPvyB6T/H/9uH4lAW0YqUPnjP44AqymbgkpF9t1gtyrZdSJjgMCOp1Hw5Y/zCGPEihXDwh4uV1wZRxTQk1iIYW3Wg33VecUt4cG4S5ygn4J4RVmJj5+yQzDHCfdoAaqOwfqIHRqLCYo7BF5GLWKxQMzgmsQ4MMtCCP3LIb0DrwDPK+FM5xbBKGPlTbTMOBjnFDqHlkLALJDBHXk5z30PmZy60tYs3EQZzwRatJ6I1iDSU3ACAGpKkWcEDwjJ+DQQOkiPIKXbCyRYXIMFzM+Fik2sAfYoFi/GH5McW8JCJ2OjkN3MTL78AWPwXYNk3TLOxoZRir0xcMipljTdihAkahsA9fDwM6+KwatShhceEtAfuAXouNGKiqpZXSCuFCt4B4Qid29s/IU2FQb8d3f3ByC7BpJkRDiHZuW2ilnIKlp25qdEFUQ8tM76SdmVsutNOXWyM42uDQTtx2bFam8Rdg17FFnNTXz1yKOb2PW//ENoV0RrIbT7dn+gxDR4T4K0AyCKdhtC5ZQgUmHZo5/+9X+tSufkqj4uOdcxYsPG9k4WyV0LNSlJoSUAcekKmOpL+oUCloO63jrVmfENCaQ4h1FidCXZOFaQ/eLGaF+Hso/qkMHN8TOHlsdc58mudHHpAwX2k4/Eobbw7Cw6F6GzEJfmQ2XhZuS5zkBCA5Jx7wPdzlPpSIfSK4UlgIfSeySdVtyQlh0x6WVWcKJUMkj9M2zlVz7J+nBd0a039BnaFRMY6hmONStc9yjwUBMFwAUmNulORr1hv6JS8aEdR7EJLS3qAsSp5OMZkQ+INGDjCm0qrY+bdfzSVHcgVoOIZ9C4sheC75ayZQkzCh3QtkAt0t7xGpBxqviFV3uqDBQdXsDbyjAvouS8yj1rJIsgErJwJA0kefizJfU2jaU6q0pyLbrJSxOXXS2/C6F9KzwMjhBVknSHH91NMr4i1G8f5Rahf+vHShGM5ZqrO4phl+pCaklkiVvBUVx/jBgi3HXZkzH4sBWlgBe3g5EKaTyNIjBRSg0F6NI0N9YXtwPTaoygqZvu7Z8pfYfyPErnhrRjE5Meo5znSSa3AI2P2KTYRltIBWl01+hWM0RBw+rRfeSrTioAT1Ugwa91QnY2OerX2VUMHnzXdhzkMse5kSJ27RMCRh8VYG3pwIEd5pr1icSJXN3is/L1DMzHIGhhWOmAAHy0JCSzF2kS9ke6MpXziwIARAkHJg3J8FCoVqMofSxDWowVN0sKEYXfErFFzEFKCRCsVj10qCKFxsKnRa7YnrIP2iWl3Wu9xG7ArwwVVSTjaDsxqaCKMnz2yJp+p60bBI7+ahTkpPONBmmCKs1jD0SDmYLL62bCG6upNxriLlJLF6gv+O4uM7c7ceBJn8LCamtCOdtoP9R7ocVHo6BbtZhVc5OQezcDcDgqrT0wAVS2CAbXFLDC0iO1TMoNRSd0UfEr1w2Epq31URmX4BiJUCYMedFjpnrmOoihTa0KgenNn7uTWCtZARjDqtPAUC7QR727dPuKZ+1NyzWvg7zHqjeCTKdR6AsKOBQ7ElessiMoLcrK0hPT32fh90clbKsKGjtK3AW4tun/87V3cF6Os7b2SVV8S2qwaD8FnQ5FEhZe4TPVYkVWZwgrY5rbUlxUfgo0wdUDUKZXz0YpUlJq2kdaCoZ0dI25n0T7D/tGkbVCj2v+dFcbhwjuM/rVHsau7c9+LJFOmUrw2Jx8js9anOrPsRKEkzOMeheZTZt5Abpso2oVJ25SdTUEYtXvdhpWglUT1xIVCgX2Mj4AyHI2Tbb56T/+G7a7+04KHUQ3Usgtca7dXdrwwhnXknrqsEFv+WAhqAFrVpnyzF2YcijSt1d23laeUS4mfbaZysmbXPHEB2sBHUodkNvg48IywxiCCF+iSzI5K1wM35KUKMJ8CYwBa0Vdd1tChz7bA1M7VA2KsOGriLZYAUeDm73rssFN1ZMFUkIDhXZ7Uz8EYgbr2Zxh5CRRlDWz0WTZ7AYkFfMf4RFSx232NeF5TD/rzIR40F4IMCxS/1m9BeCosgsKFFPsdEwp6RTIBO+6N5Rrtf4ymXQAauesNvs72Hj1fv/Tv9vYb9VlVnwnV7e2NRvNNRUurxzSs6lZCuXXPBBAOhgrlTo4BYeGJx5tnZuU7ASrLlbQJJLxCmg5S3CDsFsGY3B/keipoCXv9pQ7mfBqf/vdR0yrVPl36KBh/FZaVt21eIuG4KdSQtZRh0LeD/OScvzl/E7CF+UoTNWRU7XjcG6QQMFnZDnSo26roXenrMSa1NW5Dst5w7pHPOq1N+G1xdaW3gPHf3B4ZiciDsAANw3UUgYrUhcVIRCoyehh/0wtfgRA5fKbYpWNUIB8IEn8gmdcoRFQSsvo/ZpLuLMBV3oFSUPx8Mq8HYnFet3qvRWZk3dbV5W9iYoacTABDNl+6InLs50W2rv31r3OFl/kmR+U4RC6td0ovqrD0jcwFFpyh1Ss43TX+sCoixyhtEGuZQipJi/VK0y3tcHLBwsDKGF98PW8XE1NTXFVUQU8Tf//tG/+tCbK3cySGrT7GlpW0wLZYHUs0LCBRooSPyNv35efCW3tPK1VDoiZw0LMoCubJ64odEydFiUjxGmNEXRQIwLzv3i4WsJoZUN75SdZzgOHdrOuKhEJFO70me3mHrfhvYMGD2LFpChW5aWfOnyFByGmAdp3hFjcGuhXlHObwVRPXLhWKPJU0N4Udty9QX37/uQdKOTV0s5T4YF5hFsO/ZE9ANtxr2V7DEBBW5n3WTRGQ/P1M0yIXNqxSDAP5+Y3lFj6+vQ7lqfc46N2WOZ8MjgDo3U+6QOxNHq/ppfNXmpZsCWoYvOcUudc7i4EzAttOWHTXWHPtNoyl2sZSxg2E5dIlM6btx8+UpA90wHxeZzry1XCl/q6cumWLXALnq6L3VIZ/NS0tk5eSCArJBKUA/EF/Ojb/XW6RAFZnuUhs9IIgW993kuXPuv1wGFNrnow2DGC9j9znK8N95akSVp0fU2Lam3fEcWpoI2dJ4LAxBHIIttL+AX0JqNvYzddWWG0EBwD12GUA4+nxtZgU7oQQaA27FweI3QnyrM4z9RDDCToS/Bz4N14H0NsHlwVgPGAwkRGhC+lAL+kMxh6d0vuvJhgZRpdIDd9axndMpaJLg70EVsS6xzaWZQBuQvXRIy458AnQ3wM4oVP9s/w50EJAunByS4zGc1UXw36wwMi671r/Lh4ImnauEVpVUAauD0OXCU/uVXarHiQknXgZiaSAc4PgI1XuIELvwp+BcTDawUKLh31HFUDLvo4i8t8zHPHkrhF8Dq1uRwu8cvk3Cm2Sy5LkYTZhsX7yah33K8FoxCgVcwXrpa4EtoRqKmL0hdI+AxzlXXSi3ncpxjR++hTRNuwXsR4MMezeKlMdwx8TKoA9lnwKzC+gzJqV2Ov9sEAOgv+3bmDb6TP1tSyrGq5hJ2n7+TMMOSDeZBwB9Ke/CSV97TNNf4UTVP5nGcos0JYZAMf4uEcg7sZxqZwdxDtsNDB+eKdOrbz9dt3VQI5fqQo5HgoSeS4L0nj+FARwtG+2ncEB0we+DENJEglFeHXg8Pp4XTWx8vfoLFnGr8FLWNoWYktjqePprOZbrHTtH5Vc916MDv2Hk4r8J7L/lzVn+jz/n5/I7SPKH+1HMb2B7PpvneMlz/+/XPN/r+z2BuYDSXhHkthixYGUIC7APe0HndA9zOru45yOxFZAzBssZyMR8JRSQsYQSiovxTjKzpUtyCjcDHpgcR4OsaFqbXCPEaY6uHB0YFHIX/6jOwB6LDqoOGLuodGw61vXlbao5JcNNLUafoZztrQRyBAmv0O5CDGUI6OSZhZ129Ad7+Bh5iTdUmSskuyTerwrbpEhW4u66EbmP5xf3TWMLFpIcrowpM4Cq4weZvFMLAsHXeuYaw3HYw5BeNOCHYt5mUm0Tmw2jUmKW4ZycyQLXuUwQnth/ZhHb0D++FNZ+9pxY8miqiMJ415qFk75h7uEI3240t2XOR+Fjmew3qO58bE+euOWuv9/U59OJ0HQz709h92blRKMsxQZ5gOoNM0CsC32QzY3QSu8/QaaeOGPZk+vSY6JAV1nd9grvCTPZylxgJKEXTPQMzIjAUtWQrLFDqhRk9ZvySrGiGSWRrwGzLYokAfWPmTIncdsTTIDYjVTdcaqgzrm7aEb+hDhfBTwmSBo46SxfKQIOIUjVkCi+0IF+uYBmXlEK2jPfGEri4H406/wy6HQK2/A+q+gvtrZMO+PcDbYe1WM4Kay+GhftTzeLrgScKvxp0DdoAkTj2i6YaB9JbEcomNpz/99d+wD3QCwouaJ6VoiVVeOWWYA+qkuPy+9FhoQ7K2sqoHKeqhh7/6vUzok6ecNEp2Nh8pqCZOD/vD4XC6zhT1BZWjai59Mzm6hcQ0FR4BYHqA+cyUysyKowCPG7n423Kua2cfmqcaYOKu6E1FdoGJ33S44SLh8Qh/0KkGOkhwjYt2s9ZJC4a1zYE6GRw/Zd1Qxqx80nQFQQTsNQFvdgIx8Ta+kLe/U8tWNWKqg1vNabfxWXQJ9Mz6DIiZXb8BksVT2yJZiRPKX32Pbksp1jek2z/Cwx3NPmTyOqKI9QFJmpdurjsdSa2kWZAZYTB3XyOF9sZqQAcaLMmw7k3jZMXOzvMPmHyN1uY08q4sZstjCbTYHI+VsevaAYBHg6OBx/6Z9BA5uNU7LV+AS9JT+DjsAxYq7R+zGhne7DxYeAlbDNh1mYI/RApmClN91SRm1/XzLrrBEFHJsFnqe6Ix3OnAGx48Zmt8p7hLvymYCXHINDfh6RwEi/kxfwq4OFx7GkSYb6V6kbf1zqTsqOFQQVcIkX3WFuUBJgZWoSCR8aRHa4Ve1HHfE3NLGcbWg0fugTebddsWajabVR/vaGOpODDxqLHA6/Kp8fqyB1Leiy5GyGLDA3ivxnF4CCNP/HkKLpGNBMyuFWHfMP084FP4eV3vZFgnspudWRRlSImaSZBVaw2Q4tWeYu4HnpP72oHb4NScqq2yWOXdiCVFuRKbLlP7GU8x39PHcKKzyMXY8EPPn2O2YyjyLAFHmJ6mAViV26MPiNvxpAT9OormgXgFT01Z/AV9DqMQ7KjpUh6mPTTu8fSBnYJuLzpAvnZKSnDIJDUUJ8O3be8djyfnlUYlMCTPFmiSRhFaWwMNTrdqgqO1dIjesBWRcwmr8rYApJrUJ0m5txqKlC96fuUrDUK/b4yFyKsOScqbYjyZnwWiva/KXkGcZ9uQ3tpgM9bphISTCpCxHtJXywK4w9k+DbLZdvtIbwNdYPwO8Dei1l0I9xz4XmG3ZfR6ybe33Lj8jc/aCWFjo8a4Jb9PBSXNVCrpoIV2OqGQgTOfGuQT7x8p01EW5ZEnivxsgYLhGZJLahK9jI2TU0Z5EnjuBLf3YUDE3mP6CaNL0zEo41bB8CkdG2Z3/PTai1zwS8PMRqK23YCn6dd+Cv6L55kdnAXWnuhivMcTy8pBaj2k99GFKQKxdHxvbIBWNRr7vtDkHUAB2Yze9QPWHPT34Vrc/rS2LV/r6AeM/qmjxJTP3ejt/ufm8DgeHkL//EPui8SLauB038/pfK2ZumBOj/ctDAU40pXeH/YtVswf9W8TAVU4H/k0NVveN9qYhjrVamxoqxI+8HQDKQxMLgMyNIkMx8a38qzClz9Oq5l7ETOpXBclxXYN6847tdQb7mgvogDk5th4+ckeMc6WAhT/Cuzj8xAXRM4g9QN0u3myjJJ79oF7Q2Pjg09HpXLgFspqwGRuOqBAhzCaBxSMe/WgA/yXlDwX5GJO5/f2X9jslTzuiLP6FNE5TbCrgi9/m9r3nAUduB8P8QDPpSNvDrobIehTE7CIb+l8h1rCV7V8DuTsqxhYPvaD+wyHsEWpBWPj/etnJ0ZxDn+43988KE2KJ64LmhKwYhrvsPoCbc5jfRXweAKPY+2fRGVOLaJPHI8b4Q52K2du6gPZ+pamJDtBbLeg6aUcSR0/W6Y9GG6ZdrGCYta2JKc0WZQU9q/bHx2J2NwhsNyv2KE8M7Xe2RtYQCQ4ucaN/u4F3NEV1KCLF0kUk+i/FQkTeR7IolMidGCE0IKWqHwhD47QG2xzFzZQX+I3JZyiA3xEN034xtkdzkHwAE3vTWPWghgwEaq6Isghxr14rihjwMz3/rzgrkb21QaKkvh/j360SSeTMAUPcyDkwE9k8Q86AisHu3Y86tZ/hdqTfpSxnTbU+Q45rufKcDLlhmfBzXrCXmEiWfcX8EqRvBCq7A4WNcEc5thHSRV9oht46IGCzAP+c3SIoi/z4UNVB4125sRn7G64373fOp8ElBkOKi64xxpLnpTo/BCA4ZGYfRvUzqHFBvYjedh1LI8kKvQuRZZE6f3G9g6LB+CsdO2me4j4yjmlLYuuaj7R9CmncUONp1t6aSACZj20ARV9+6hARYkJdHlDMNa4dz9sPIch4kf3QAKdSqoy42TDQV15WOnDa2ZSpSN9HOouwqhNMrV3oub/jgaF5gVGnu/LY5K99HCxMERxSoRPwYiWlgu9Zydvb1k+CoNma9JKVTwtBBb6TsXwT4FWqDbjvaWDlgxKuoxkEVQykeWp5sPBEITD4LDP+v2+rHsGCATPFHwkzLxdO5dxV1FBHVFBgCWATKhGKjMfVvp5XKF5rDjicVWx9ed1SHOi72X1NpvJcqojdqhKwlp6NAHV8dq+TjCwhkqfGJjfgZSLNVvxNxZlxVMWcskwuUgt4wSX72yyf9a974KpFdenZ8z4snvLQO9pWBaHTrfJqH+R+7AgGEXzbsOTwhXm92yDqCckaxbdTwr9VlZENKOYPJbAkrXRaBXvZYNXDnA2xqoqcqnB6hpLqum704OfpZZLBvwG01JcED94xm5UkR9Y9i2RW4jsiipHC5ZFHtIoT1mtiuLOrcdUG/RK5ZkLA6gs0awmqaovfvmHkPJnCMJtPKGP9W5baVothjmmDNNjN0OcZkpPyuolRRCgvbSVUQk7gEeMtpz/GeYVzLd0QTy8Pc+0Wk+ZskxlFnZ3Mjo62wgXq1Zdbqc1PBUjyy+nJlbYoHBlOp6QBX1WmoCfBLTZaKXUgiTv/FTGkqJ0W6Bk7qxh9sTFNDL/M62/xFu63IK3OQ8kgNeok5Mr7TR9jFwMS0SyiCPHCk9BpMoa6vzke3CKLONGwQPlcR1ieIlqXMk8eXGhGHsj0AVmFcrBvgLXnE75OG6UhxkYgFglF1agJPp/9fa0cqrkLkg/WWFxEBSDo51fQQTTYfSa7bZPKTYWq6sOzFEkM27wMzXIKRVBN3/6t/9l0MfzzlT79hb+luUFaqMbYM7Yw4O18Q30+PaHpfdHn2/pg8oW1MHr6ubyd72Poe4Da6DfBw1qPM/JmgFdgWbAXfyAu6yfLOsCjb/Jl1OcwXD/oMDAB7H09Vlw4VLdrvEtIYSidMxWjXQSCJ5h+aRtTKuqP9QRjCXkwUnCKiLKLTgsoL6CJmTgywpOGH3cAl/WhahB7w2OAC790PAxt1CrUR+leeqzpY/SIktQa5s//n3311mKojBF0047wPE8Qkky6NP4Dg76hZWG2w/6w3v7GnX6el+elb+T+lTlMWoYPFB/MQCGpV4bmupLslJuGJMVsu9nPH0dwZqZ1UOZ9zKZpo4+g3uLgKXHGEMDHWdQvM4otdyrymHQzT3dOYrTDJ/cCvkXmUe3Ql9Tt6eUvI8VhgD5t34uD8BW941u++K+Gm8bU6+P/rciYXlKPi0mD+0xmQl8B7MhKUe2YS4/Y79KpK6AlfHvsFP1qLo1hckgbUob/6pEnjZGuaEw6+6uskl3d20GeMF61om/FAmVprijnyqLZFhq+6Xi815h6UeqWlCNKKi/vnFXOwq35pTx9KhF1dR34nBbEwz49A47coOicAfG38tSH5vFxdLBaiQSsW8+vvvaXCvvd/JD/uWPjIPxKFwB2qBSbu8OltiwWqlCjqmImG0bVeHJ33FQrTVA7jTCtuqEaqTF6b3NI8WRbRhoS+fAaWuiWbH8C2AZnmCVdhO4X84C0y6/KqpasBQEp1QFd6KyVhGzmdhafHddFZN8XQ76jf34d3U3+8d/vJNiuqOBsKI/YyNxQ2Mw1+V6uTl4tMVIWzngxrYAeq093O1g2mLM6mQIFgU9/Q4ErDwacVsgfKnjykQe9YMzFcG3jDZpLHizzTXUZ3HaZf2naKoMF7QCTcNQtdLkVscMC+sGV46MbsrjCI3qSzBBsBIshgpC2/nYru5l64N6ujUG1bJKV7ID213wcC7MaqcW078n7dXSgJObJaJQX9EpEGQ2i0SjpeSYJSWHpSjJknRglXxnKU1CR17LSZiAYVmI1SxLeLZy2eRPVNezdLV+1eKedZv7167wWd0yUxU+dS3Ps63YA7osl0wuoF4XnOwZHt0J3TwB29m9KqNHO63hiLIhHSdDZwHLgImErPLdXar/K8E3RLKd5i4e2DSrVajU6CaKkqqju8+42D16JrJW/ZYk/ks6K3tT8kNRd1UAWaqOJDyrnIiuNA74cooHhs1K4Q+rPBXe1QAmcxw9xXQqvAoPbXkQ3ixrR+hPylmWH0wrXVPBG4A81WxLjoalvYIzegWfW9IU37AwlQPYFdjFEXM9GLKAqycjl5GsdPDzEUAwwNHJhQlyloP2dW4jVMzEIRGu8xeUVEVQO7+82jqVy012nA8v3//25ft65fEZSCQHU4RN/FEe6Hzu418ywerXVHEc9z1dEXicgKGxlFHZrWmeUoQR7G/aVwv8abJWKTuN3HNRliM9l1VT/BCTj3Bb5kJMq3YWRlb0WKkq6XlLARQPa3hhJYd6Ha/Wkj8EFCggSkXDvdl83HetVkXWHJWR5WEocEe8HByWDWgfWGarKtmZHgOiIkZUJKQaEVMWI3x9xQ763UYmo8ShLX9JxyC9w7RTe+pjQRijb9N/GOlqMU51Hew25Hz7gapFrcOuVczREGD8irQCnofuwpHLTNMbPzp6qCdGEx2v0V/BQfCuzFIv+UoCNelgSkLFKceVqanH1JfEp+xdHVvHg1ZiUx4Jhq9FmDmyvKSDiazFZ9GFIxAH6oHONhq0RTkpGC48hw61jCfNshjV7QslMDRVTTRLUOlbZP11UUA86xZ/mdQrMvgvRe2PS1aaGFvq0MBn1dK79TIGIKLmghKcej/Qz29xT+dySw6OsciyOB3t7c2BZPOp7UbLykiqg9oDI0PwVKR7eDYgzfbQq0fBu7fZSTcq3/cAUflljy+9hwcoQSnRuVE0CObqLkB24ywt1o8eHR7WpCs8rpMqyCH8c6UpSGaPCpLnKWaTcjywrOl3UC+wppvXi9vgk7Vy0XkSpHwmzOPuPyUDVGme8TxbjE2clVWMs2XHG5s5SyADrJbblsV9d175RewhFSP5stLBGR4up0aTaVB71DgGEA23VJp/TS5KPqlzULVEnVuv5nFKBY8mREClvDd6vTDq4Z+nkkV/5CPo9i7e+YyYBHhkMHxE6zYYXeNS3dwluC2rbYyrI3x7+hJtVHA5kurzDx9fAFIrVTsqxdyxqElZrrDUM/RnQeqFDDP2hD3sN4rMyYOw2JMq/2EjGQT1ujdkgGJtSwGmIMYxzKQQDxPe+9zvHffOvvrehuUpV+N7FBkYC8UuZF38Zg2w5boikgu+tPHYRGy2+PRgk/Cy3vGmAsLa5JtQ/X5Uzw6q58o8z7bUF64TpaKVM4Wnu9SokOWRjfLPTjcqUaDwLawMnPJaEVMF4btU/90XEWJWPJN/+Dqq/FETyrNPZJp0ZNvVvxZAUqZpK7ULpDWhRLJGCoJ2UbNZcrSS/v11aUkONDAaEUreliVakxtKLUBrKfnLge/8X4qs01Q=", "estudio.html": "eNq1W91y20h2vtdTwPTMELBB8EeiLJGCvLassb0lr72WZydVKpXdBJokLADNBUCKMs2qfYLcpCrJXWovkqpcpCpVudi7XKzfZJ8gj5DvdDd++CONZyaZKg+B7tOnT5//cxo6uucLL7uZcGOcReHxzhH9GCGLR26Np7XjozFn/vFRxDNmeGOWpDxza9Ns2DioAVgOxyzibm0W8OuJSLKa4Yk44zHArgM/G7s+nwUeb8gXO4iDLGBhI/VYyN024ciCLOTHp2k29QNh7D47aqqRnaM0u6HfXiJEtmg0BqPe/Xanvds+6DcaExbzEO+Dtt/Zw3sYxLx3v8M6/u4+XrN51rvPD/gh53iLpng7ZIc+Y3hjnte73x10B8MW3sQVsAwP/f3BcufBYiDmjTT4FMSj3kAkPk8aGFkST+yB8G8WEUtGQdxr9cc8GI2zXrvV+rY/YN7VKBHT2O/NWGISpVbfE6FI9DuosfpDsKXX3pvMjfQmzXjUmAZ2g00mIW+oAfucjwQ3fnhppyxOGylPgmFfzHgyDMV1bxz4Po+XO/czMVlMRAo+irjHBqkIpxnvYxTETOb9kA8z9ZRoCvHoB+kkZDe9Ycjn/RGb9A4wyMJgFDcC7Jz2PAiMJ/1PjSD2+bzX7RNk4zoBKP1vueOkfLRYQbNxaikSq6/41mvTQUUY+IaaJQHlk42E+cE0VbRNmO8Tv3cnirTOZK62MwbTLBOx7QyyeFHZLUvAnglLQPIKlyHlYvfW2k6PKhvh2WhjFyWRIB6D0cA0TVKgmohAcuJ6DMY0sI3He7EoeaCJcsQKSfc7Xme4yzaEvrKmB/axQcj9hQDaILvpObvdfF+fD9k0zGjBZEH87bXx7PulqBMesiyYcTl8vM6UrxfBFhp933AiHk8LAcci5v1NJVMa1VLK1t6m+b9SB/bxHAWxcha99iHNRWze0Na235qNS4tg00xI4h0x4fHaCQah8K4wS4MGs9WvksMqTF/vRYac8XnWkGbRIzPatGEJ4HNPJExyRvFJE39AekUUrx6QzlThk1xSaOkdKlgQ3xvTkVeOoIa2aGC+KI1YGC42rAN8Y1lPihGehM8nt+iRuCrEVMVBw0Rx4zqXRwtoBgIkRVt8kproSVuTbqlLPCbrHYokUnYMneZ/ZzYwY1W8z4qjka7IDxLuSfQgaBrF27xX7thIY5RUD/dn16CQxUGULjacIElm1c/1P07TLBjeNHQM05hzFLkC/Wq7q7iqXE06a3bwky5qjShnG10IdVsFKcfVwP3hcAhUQTwUW0SoogmClrT5faJxU6lIIxA0uZJ0LsW9Pp07N92208UuIR/x2F/c5lnkeq01j9Zsf6+7bvtVq9KJgOf9YtdTWO/acVZcYnG24iyGH8y2a9YWBdXZA9AarWWOIFjkHgiLirRiw43sVmgJYsla7ePuw6i38TSIkazBx6xQt4Wq7Tq/Kef88LvLHY/FM5auO9vfRNwPmGGW9re/h4NYix1DZi05Z0iVDnKfcFBkKgcU+I2NyF+1icNV+ezKFUp3CanSCqU/2zeounh9zgq+donPGCQr0ZDGtaPTG+y2FHQmRJguVn1OoaiNuQ5TxqoHyQ/V3aJ1+lRpxkZ8wPwR3zhbu9ta41puWSsA6ydb7hw1dU59lHpJMMkMyvzdWhBR5h6xSe14oV/SWm9Ry8YJ57VebZxlk7TXbHp+7HxMfR4Gs8SJedaMJ1FTAv2m5bT3W06rOZgGoa/GnEj405BjRc1WqJo4tYjT5s9DyecsQpacNj+mUbO2XOIQknicoqmqE0rN8QZTNALfrYFdKCwMQw54IUtTtwatqslJqlNSOQ0AJQ3DZxlrzLCQz2v5AhHXjt9BWaYJO2oquON1eCShYNTxm4B/YmkBtRW1ipgEzFOR3ooRm9/Ujp8kXhCGt+87DgATZqy65VETp904NRixdcL3FTdCoqpWbKBnYXa147/985+Ms+kn42//+N8lIRUUlGds42NIhaMs5mp5Vbf9IAQ4hwUGIgGkfjJMeC/rDl5iFX7gGT2OZU/yx1u38BMWIVnzWO34GR6//Jmeb4WGvceAfEM/hpmgdE1ZYmRKD0qyFE/v5KzHomg7Y//nX/7+X42TL39GNGA/m7ueW2vZrdrx91T+8O3nAEz7QEKdooAJfXYr2KGEOmO+MIJPf3TuYDuAGxVoBCbnVqy7Xbt9UDvebe7dCtKyD1qk50kwuJ26IYqv2vEJq1rgTzMeWe12vp/OybFBnHfyPMehBLCyX/6T+xmKu5K+EYvBE7g7Hoq//enf1uFKT14zpP8FsXcXWuTwu2XgWsmruit5j+5jlLncZg65Jc9cjTZ92fUYQ67XvZbRMtqUC2nMBweFBynPLWPN5jjFzc1RFZJKh0xjMhbe4poKTy0D65oN0HgishUv/QSRsYFBtukEpCFm8BMnsJakArCB9Br1BexehF/+YyTiLR5aCvKKh2RUf5zykGdiO8wY5MHG/+EvxgmbrISP0kvnPytRWIVLHFhFYOOBwVLj3Yu3p6fGMBGRUZdhsd7P5xevk0GQnUCUCRi1rALlgdbTk80VUATcCpbnZ+++P4Mmo6LYhiKUU2mzBFtd/uztk5PXX7G+AreK4K0Q0Wk8CxIRw+ayrUh4OZ821xYobDs4apoZ37ipe+wLbypnIKfk5hyy8jKRmKnV11Dfvzw7PXcXcOu9i7oy2/fXfOCMwkHdzgdIR2ESq4MqEgSf8olLW+YAQDORSYAe1NGehinc61EK7hhiKrrrwXFANKBaZ+8ZQroaXeaU/u7lyam7WKexV39+9tRA7OaJMDBomAhtnjAeGr99c/rc2n4GtUiPGCaW2cZz4YvMNt7JdO1jWlkJx+MMB3Os+v7p31FL1+C52hvmD3GQ3djGD3HCWWgbrwJkaAKLdwy93BmE8BBY/JR+OaK6GrFu56QiLw0qG5XASCgohgtHDD4C8vXT3wIqk7FcVLZ9D51KeApXuus7aRYC9PzdmaFHv/xXbOw+Q3kSVcgIhd9+321pEs5eP8vZ2m19uwLVed/uVqCi6U0O2e5+K2moaECvrnJCAypujKfQAeAqVaFXf8UmDEHUkGOYq+oEiIY3N6hHwoiEVd2gtRgwaMDAsRAwPHk0k3iXcB+vCR1wFvhcvEcaEsQscaLJHpb+gcaUDshFBdgI1rSLbLsKhve//qeEAPffTygZYs4kHgHgZYSIFtMB5KShJuvLwg4TKXaeuDG/Vj7M+ZEPnp+91ePmgsV0ERCwtJclU26ThHgy41Dka0Svp9PhEFU8TS1htjk6FIfZm2DOw7fUfzNfsWzs4HymumMoZ+yOZfWri84R68wgjnnyo76HwOMLaaVV9GKaTabwkwim59T6dRXp52+fPy0HS8Rw6hySnIBgDfnk5PT8+yCMAu9dOVehRAZZDDs8lq1gl07Y3yk8FlUyDlZhwckYlZRZLPXh9EJOQIUbS1HB8gqHz+ndtPpy3CmzhAqIPIXZmqt7lALTJEp4VAF78+rt6avnwJIw8pw5EdQzInC4Y5RGrlzlkLtWO9P6NedsWjZKuD3LUebO8w2hltXtECAm1N6b8RMWYVMTKWQprWZFWoSubXdarYJ2LwslqpUQZ2IDeyvvAK6Z/wxlJUlOioCGqcR+izwi49vGzicc8mo7nXzjURJUWfscry94OIFyt1v2HnLleYfvttVDp7PX8XLJIKiZtLg4wjAUomoqr3g6NivsQTXCn3MR8Sy5MTstHN+yK2LXSkUFFAvN6u3C0pI9X5E4lB9RjunM3YY0mzcvm518DgUUWK/wqKOXhEqQgtIxj4IKoS/wmk7GYPIZCQeKNZT/4ci7+G9vz3b2q6em5aX6TuMKqmd5g5eFa7g6wDCNHY+l2QqJGFP25ERsQgZOhg727B3Y9L/qvoAttk2C6O5tDw4Yw7btKgIsAoIdRCXDm0JU0zC0I7gb/cgkitRdLBHkhXdVtTh6hxkgx66MPhXzXQy+gD4d2JRWKs1R2CiPJlRsxgKl2+c4WHEAlVHJ8TIpM3Pb9CkTkJOVjAuzcpw49Ix7CGrJGyiBWf+lbZgwGKRNibI5CrNhs271FVlyh8rGEoZYN5zGkkmGbDmYV9QVNBTJZ+5Cdwx6F6XU7YuOvWfvXtqlRKBNdttp2ST/SztvHuhFHd+3dx0s69pQAFo2GPoHWOZ0beeQlqpllPoXHQS1lrX2YVSAuGjsYWO5eL/1qEV7AiMMSf5T64uGQoXYXexxsWt37Uab1g6H3e6jRzbJ1gHR7a49ZGGKtbK9UFnn7NkXLSzrqlXKdLBuX+6rFi0vri77IFraALlvYvELPjfPLlqXyjToPiBGDXnjnl20L+VQXlNKm3Ac5+yic2k50TTMgkl4c+6xkCXmCwTJIFrHunuphqtY9yQJNLqCuNF58MKmf43dB0BG5r2yrHupbaiSwGP40eVjFUJ6pO+bIC8rKPblzsHQvEfrrMWWPR46+9TcLDRsAFszrQUM1ZlOfPhFOLwkmP8oEgRUEiJdBMzhNa74aTTJbgrbGWxYKO1NeFA8zhCkuCnc4wWIEU6Qkpe28pfzKwpSvhqDnV3JDDbWBACRAJsjJBf8KYVkSnGIyP4Aw5MboBiUwxZlAOGNonoPc1FJv7WUpSuHZmCbkQ4KP4G7APvqTYg/0xjMNAfWckm5FKI3OCvmFT4PYQncZJ9c2AYP3XZH2rTivuao5xKmEc9OZLO9EtT+IIsysNiy0xxIZmnbQPovXJXrsbmZOjc2cgAlGuXyqpNzmwBS55P1oO0cNuVMxmJKCJyhmD3Igx+yWwsRsEDCXPapmKT2GXd5WB2QoGzNADxn/lDCIPk2maXgPZGa3Hrg255zU87qkU8PC5h1eHUkJB3wTuCGkh5d1GHXmLPE9ZtIfOTrkN4edNWL0rE3ifioQpmSqalynUIBqxZC5VQKE6F4lrkte+a2+nep+Xff3RPOFAn6M3hOh3olMC+VBLmlevVnD92RwzLsPoAypiWrPOhd1s9oWjaxHutfNdHc7d2xrLlLCljI6SuUBbDfmHV5jVO3HJU8vnt15n74ZiH5LZNiM0NSKs4EfYh0jq3jkWktDTx8+XM8moYiNf76F+ObxWwL0OzLvycIADw9GiTH3yygcwD6HtmAb3Yw/eWfDBq82Tb4aWUw+vCQrNkkSh1ZJqaPP6h9K0NLVUGmH3r1OsyfpTexZxSi5HNUYKk5tBaQgBZK4rJrFiCv5Jk3Nof2AuIZC79Xf3H65Fm9NGcUPFdLjxEQBKoHZdRZLnfWN0rH4tqk25NK5PZY7KeubKdc0NRlnzRqGIRcZjJ91LCmTnENMVTgFvRKZjbOmBHd1kLCD/uDhLOrpXb2NGYpinJ5UoIBecoOqqMbqG6dbjXrpXqM9NF1NkI/T+gcJiGETUu15pBCxkKrgpXqkxP94dqHzZausaI67VbrAZe4ud/MkS2//aD0FBvAlqyFCmsokcSMy5ElZspcbw2gMtFfSwiXKgjBeOSK/t0BSWxkymJbhi9QuE2RdUUn2AGFjRR7XwYXoCNhuq5bp5ZV/fPn/HUc1C0ZKWSpsVaxnMPN+sirykpEdaWRiB0OPG/fBvNG45inac/pdm36djBUb62u4tzK1rqHVqcTffWO4AkkKatc3ViobPpoiVimdiozexJMfz1sRbH0MlEQOzeS3YVLumm4UazrpnIQ7pMKutURIC1LhH5ZIvRJ7WQHfMU71etaeeALqUEjwZ2Qx6MMUtWYiuM/yUFe0URxitwMZMvfvai/5SCJmk8nqg9ETyJBTYyHcxZOfaafMvnwFGYpH16xWPalXqBQlgNvRMrfv6tfqk1WSExFkpkmsweWe2zKjZVzfw07d+jjUOvh4aH17eFhY212sDKr6S9dhkcuo7qTtdBMvPDkyktXcsXxwmDyRM5QtJRYcj4M3KK14sG9ZFz3Acy66snXKUuq2r7CDIOBd/G4WX+PsxsSSsTYB6WdiWOS6zEVqLUizmrrZqBzNQmcU55L5PJx/tSrHhEpvcaqjqajNzU9JDZyr3Cb53AQ2mJvcwbVhDRfodtpRDC5GNCbf5Lo3tMw1Ul5x3KG8JL7qLpkGJkPz35EEmASMGUDK8B06cCCOFXgRThWt0YbDnzd3EtnIzvr9cd1+XVJvVenu7L6BrbSgNYxPa4fDYrr9mOK1SfMZ4Z0SoZrTGMVVx2aeSagkziWEfFPOElsGyEzZOeVBn0BK3DqPSnOdfJoE9XsRayAn+bJhG6GjusPJcSZJNS0esq+Vc6MallWwRDemepB6oysf3ugK46//XrjSRhiqfy6wch1GwlvcsoQ3Qfu8aAioUyMRqESpz1w6LoVAnVmOJWM76u54sopdIKRcq56N6/YhJzmL0odI/dJkrAbwMlfs3Tz1uPyGRbRK9+0i4y++y6SdgLsRItMJdSIZckBSs7VgF2/X3+YF7kjWeTmqZylo4HOfC5QJ9PiS2oomeZFbHuXMPYPR/IOOMjvbivXr98svCVdYwZIA+OlutX7YDkfBfJ9mazphhF5axUGKnwltxDnzED5oT1EfNmnBIjl2Y9KJwjDd9/ph3uuyyz9jFLA56+nKPJ3rT5zqI2emZYcfRnToCM3kkKSZLCf0qHqh0pfq0NVF+q68Yb/BKsr1Yf2HiKW5fM9Cl76sIoZ16l7cXmLVq2q0jWqhWk6NoUFSeIlJ/baPb521DfiufO6tm6t5u/lyvr58/oGZQoqFW+j1De/Rod7F+XzZcnRyD2O5C7SKbhgh07Ml8V2hX5cr7eHy7LLXp15ytLA28zBDgYHg+HQLrZT2VHlW3o1UGke2z6fZOMfkyDjPVUUWEV0vV5lk0oprx3VcX8tk48OUkxKsK7JypZF71T6O7euG371yt3HpirmH7qva2Eeiimd5/CTYoLid8JGMoaWnVB3oFVAB309Llx/e7RC6Fbh7W6SKtTM3eP5tjApMVnSjoVV3Y0YoqfJ9RQbYfx0hgcComsXsy5PWLcp2/g/Ise6k9cX6hOoy9t4TYQsCvmVUSNcD2XLr9nH+4l9Vm9jVKZDoTER2Z2JSUBZZU6aR2GaPiJCDfGzu1IaNHM3Jz1nbqsG4hz1wIsHTusRNXbyBHy9R5QBPHNu8O/TQwB3u6oxU23yZGu9mtINKCoumI1st3IyB0lBgCzWBjcoWP1uGg2octQtOZv/lBjuzBSqspBlfyVJIDUqJXGX1O6tvm+VXzWKrIJbS7WPlyVr++Qp1E4l+1ylQ5l57N77ivy0vxWmShf55UrI0nTpHPmOjTey6C0bb8Csb7xTxIFqU6C8+9N9Zn3xKMvAErAIOo505e/gdLUtbYOhb8rVn+x1qldnlb6E0u87uw+/Mh4uIqcSklRYiRwdlFx5oymjie6Af20Hpf//TXROYbu/egAdN4nmXG/Gm4ZTXFSrB1Mey4Yjyb0Qu7WMZXXK+sYI6u6W626oE8XoH96emfWAvtlo0vcbtMIX1zHVGW7dU1+pyS87MCHJMknLN2MSfUXziaugtCA/x+S1vbv9qv7u3vTXfp2x/vlH0fg4EwJJeml2Pop3qhTJrz/jYcZMXUSjVqjW0v6ms72N+7T3jin7oHIjqB01WeX9rGozmtVGaz0JRu9l4/ZjKktf+QteFi3W0qOjtCxudl8PiDnOjIWoR1UzmCXeOJiJ9PPnhc661lqpG9+M2b/uQza7+vlS9Vul1W+TLi3sqr4L2EaVPskVv0lN+nTNWgNX3II2REEK94JARDUXflXJpTvOyjwVbNHetlTPWOHSPPml8a3seqjWdSW8XTopsmoTNKz0qCsXEJFLZqy/EK5/HRc2Wt4/w6yHVWull7LjQcgvhpcPPxhH8g/+UIUO88zAwaEnsBCL/l5CTn6Ab6r2p5j0o2u0SzKJ3nsa+a+jtVoXDrfur7D/XhoD/NQ5J9V/wxDjUxPmzHQaRaOS11RGzNzfk5Wb9Vnd+vy5ntENAJGsLyJuk+HMrUpGAkMxgtg3r9xjtfbqtrXYRvbCdUDW+6u8gdLMdt3amkiohF9np/rrqSwnn2dsQtyjK4ksZ/RA6lf5zfhGi5JnGFhrDqke2bIwMXVZQ40cizKl4k9mjpryb2WOmuqv/f8XT9EttQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
if CARGAR_TEXTO_A_3D:
    try:
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
    except Exception as e:
        print("Texto a 3D no disponible:", e)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()